In [0]:
from pyspark.sql.functions import *

In [0]:
df_trans_hospA = spark.read.options(header=True, inferschema=True)\
                    .parquet('abfss://bronze@adlshosptial.dfs.core.windows.net/EMR/HospitalA/transactions')
df_trans_hospA = df_trans_hospA.withColumn('datasource', lit('hosp_a'))

df_trans_hospB =  spark.read.option('header', True)\
                .option('inferschema', True)\
                .parquet('abfss://bronze@adlshosptial.dfs.core.windows.net/EMR/HospitalB/transactions')
df_trans_hospB = df_trans_hospA.withColumn('datasource', lit('hosp_b'))

df_trans = df_trans_hospA.unionByName(df_trans_hospB)

df_trans= df_trans.withColumn('Src_Dept_Id', col('DeptID'))\
                    .withColumn('Dept_ID', concat(col('DeptID'),lit('_'),col('datasource')))\
                    .drop('DeptID')
df_trans.createOrReplaceTempView('Transactions')


display(df_trans)
                        

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW quality_checks AS
SELECT
CONCAT(TransactionID,'_',datasource) AS TransactionID,
TransactionID as Src_TransactionID,
EncounterID,
PatientID,
ProviderID,
VisitDate,
ServiceDate,
PaidDate,
VisitType,
Amount,
AmountType,
PaidAmount,
ClaimID,
PayorID,
ProcedureCode,
ICDCode,
LineOfBusiness,
MedicaidID,
MedicareID,
InsertDate AS Src_InsertDate,
ModifiedDate AS Src_ModifiedDate,
datasource,
Src_Dept_id,
Dept_ID,
    CASE
        WHEN EncounterID IS NULL OR PatientID IS NULL OR TransactionID IS NULL OR VisitDate IS NULL THEN TRUE
        ELSE FALSE
    END AS is_quarantined
FROM Transactions;  



In [0]:
%sql

CREATE TABLE IF NOT EXISTS silver.transactions (
    TransactionID string,
    SRC_TransactionID string,
    EncounterID string,
    PatientID string,
    ProviderID string,
    VisitDate date,
    ServiceDate date,
    PaidDate date,
    VisitType string,
    Amount double,
    AmountType string,
    PaidAmount double,
    ClaimID string,
    PayorID string,
    ProcedureCode integer,
    ICDCode string,
    LineOfBusiness string,
    MedicaidID string,
    MedicareID string,
    SRC_InsertDate date,
    SRC_ModifiedDate date,
    datasource string,
    Src_Dept_id string,
    Dept_ID string,
    is_quarantined boolean,
    audit_insertDate Timestamp,
    audit_modifiedDate Timestamp,
    is_current boolean
) USING DELTA;

In [0]:
%sql

MERGE INTO silver.transactions AS target 
        USING quality_checks AS source
        ON target.TransactionID = source.TransactionID
        and is_current = TRUE
    WHEN MATCHED
    AND(
        target.SRC_TransactionID != source.SRC_TransactionID
        OR target.EncounterID != source.EncounterID
        OR target.PatientID != source.PatientID
        OR target.ProviderID != source.ProviderID
        OR target.Dept_ID != source.Dept_ID
        OR target.VisitDate != source.VisitDate
        OR target.ServiceDate != source.ServiceDate
        OR target.PaidDate != source.PaidDate
        OR target.VisitType != source.VisitType
        OR target.Amount != source.Amount
        OR target.AmountType != source.AmountType
        OR target.PaidAmount != source.PaidAmount
        OR target.ClaimID != source.ClaimID
        OR target.PayorID != source.PayorID
        OR target.ProcedureCode != source.ProcedureCode
        OR target.ICDCode != source.ICDCode
        OR target.LineOfBusiness != source.LineOfBusiness
        OR target.MedicaidID != source.MedicaidID
        OR target.MedicareID != source.MedicareID
        OR target.SRC_InsertDate != source.SRC_InsertDate
        OR target.SRC_ModifiedDate != source.SRC_ModifiedDate
        OR target.datasource != source.datasource
        OR target.is_quarantined != source.is_quarantined
        ) THEN
        UPDATE 
        SET
            target.is_current = False,
            target.audit_modifiedDate = current_timestamp();

In [0]:
%sql

MERGE INTO silver.transactions AS target
    USING quality_checks AS source
    ON target.TransactionID = source.TransactionID
    AND target.is_current = TRUE
    WHEN NOT MATCHED
    THEN
    INSERT(
        TransactionID,
        SRC_TransactionID,
        EncounterID,
        PatientID,
        ProviderID,
        VisitDate,
        ServiceDate,
        PaidDate,
        VisitType,
        Amount,
        AmountType,
        PaidAmount,
        ClaimID,
        PayorID,
        ProcedureCode,
        ICDCode,
        LineOfBusiness,
        MedicaidID,
        MedicareID,
        SRC_InsertDate,
        SRC_ModifiedDate,
        datasource,
        Src_Dept_id,
        Dept_ID,
        is_quarantined,
        audit_insertDate,
        audit_modifiedDate,
        is_current
        )
    VALUES(
        source.TransactionID,
    source.SRC_TransactionID,
    source.EncounterID,
    source.PatientID,
    source.ProviderID,
    source.VisitDate,
    source.ServiceDate,
    source.PaidDate,
    source.VisitType,
    source.Amount,
    source.AmountType,
    source.PaidAmount,
    source.ClaimID,
    source.PayorID,
    source.ProcedureCode,
    source.ICDCode,
    source.LineOfBusiness,
    source.MedicaidID,
    source.MedicareID,
    source.SRC_InsertDate,
    source.SRC_ModifiedDate,
    source.datasource,
    source.Src_Dept_id,
    source.Dept_ID,
    source.is_quarantined,
    current_timestamp(),
    current_timestamp(),
    true
    );


